In [ ]:
import os
# Version for filename
#ver = 'GRU'  # GRU or LSTM

date = "mother_folder"
dl_folder = f"base_path/{date}_DL"

dl_number = "project_name"
#os.mkdir(f"{dl_folder}/model_{dl_number}")

In [ ]:
# linear_svm_train_save.py
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, roc_auc_score, hinge_loss

def train_linear_svm_and_save(X, y, test_size=0.2, random_state=42,
                              C=1.0, max_iter=20000, tol=1e-3, feature_names=None,
                              use_dual="true", class_weight=None):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    """
    X: numpy array or pandas DataFrame of shape (n_samples, n_features)
    y: array-like of shape (n_samples,), with labels 0/1
    feature_names: list of str (optional). If None and X is DataFrame, uses X.columns
    """

    # Prepare output directory (timestamped subdir for reproducibility)
    #os.makedirs(out_dir, exist_ok=True)
    #stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    #out_dir = os.path.join(out_dir, stamp)
    #os.makedirs(out_dir, exist_ok=True)

    # If DataFrame, pick columns as feature names
    if feature_names is None:
        if hasattr(X, "columns"):
            feature_names = list(X.columns)
        else:
            feature_names = [f"f{i+1}" for i in range(X.shape[1])]

    # Convert to numpy for sklearn
    if hasattr(X, "values"):
        X_np = X.values
    else:
        X_np = np.asarray(X)

    y_np = np.asarray(y).ravel()

    # Train/test split (stratified)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_np, y_np, test_size=test_size, random_state=random_state, stratify=y_np
        if len(np.unique(y_np)) > 1 else None
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),  
        ("clf", LinearSVC(
            C=C,
            tol=tol,
            max_iter=max_iter,
            dual=True,           
            class_weight=class_weight,
            loss="squared_hinge",
            random_state=random_state
        ))
    ])

    # Fit
    pipe.fit(X_tr, y_tr)

    # Predictions & scores on test set
    y_pred = pipe.predict(X_te)
    # decision_function scores for ROC-AUC & hinge loss
    scores = pipe.decision_function(X_te)

    # Metrics
    acc = accuracy_score(y_te, y_pred)
    # roc_auc: use decision_function scores directly
    roc_auc = roc_auc_score(y_te, scores)

    # hinge_loss expects labels in {-1, +1}
    y_te_pm = np.where(y_te == 1, 1, -1)
    hloss = hinge_loss(y_te_pm, scores)

    # Save model
    #model_path = os.path.join(out_dir, model_filename)
    #joblib.dump(pipe, model_path)

    # Save metrics
    metrics = {
        "hinge_loss": float(hloss),
        "accuracy": float(acc),
        "roc_auc": float(roc_auc),
        "n_train": int(X_tr.shape[0]),
        "n_test": int(X_te.shape[0]),
        "features": int(X_np.shape[1]),
        "C": C,
        "class_weight": class_weight if class_weight is not None else "None",
        "random_state": random_state,
    }
    #metrics_path = os.path.join(out_dir, metrics_filename)
    #with open(metrics_path, "w", encoding="utf-8") as f:
    #    json.dump(metrics, f, ensure_ascii=False, indent=2)

    # Extract coefficients
    scaler = pipe.named_steps["scaler"]
    clf = pipe.named_steps["clf"]

    # LinearSVC coef_: shape (1, n_features) for binary
    coef_std = clf.coef_.ravel()  # coefficients in standardized space

    # Back to original feature scale: coef_original = coef_std / scaler.scale_
    if hasattr(scaler, "scale_") and scaler.scale_ is not None:
        coef_original = coef_std / scaler.scale_
    else:
        coef_original = coef_std.copy()

    #coef_df = pd.DataFrame(
    #    {
    #        "feature": feature_names,
    #        "coef_standardized": coef_std,
    #        "coef_original_scale": coef_original,
    #    }
    #).sort_values("coef_standardized", key=np.abs, ascending=False)

    #coefs_path = os.path.join(out_dir, coefs_filename)
    #coef_df.to_csv(coefs_path, index=False, encoding="utf-8")

    #print("Saved:")
    #print(f"  Model:   {model_path}")
    #print(f"  Metrics: {metrics_path}")
    #print(f"  Coefs:   {coefs_path}")

    return acc, hloss, roc_auc, coef_std, pipe
    
    #return {
    #    "model_path": model_path,
    #    "metrics_path": metrics_path,
    #    "coefs_path": coefs_path,
    #    "metrics": metrics,
    #}




In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, hinge_loss

def eval_on_validation(model, X_val, y_val):

    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)


    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_val)
    elif hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_val)[:, 1]
    else:
        raise ValueError("data not found")

    roc_auc = roc_auc_score(y_val, scores)


    y_val_pm1 = np.where(y_val == 1, 1, -1)
    hloss = hinge_loss(y_val_pm1, scores)

    return float(acc), float(hloss), float(roc_auc)
    #return {"hinge_loss": float(hloss), "accuracy": float(acc), "roc_auc": float(roc_auc)}



In [ ]:
import tqdm
import numpy as np

for n in tqdm.tqdm(range(5)):

    save_dir = os.path.join(os.path.join(dl_folder, f"model_{dl_number}"), f"model_{n}")
    #os.makedirs(save_dir, exist_ok=True)

    input_train = np.load(f"{save_dir}/{date}_{dl_number}_train_features.npy")
    trainY = np.load(f"{save_dir}/{date}_{dl_number}_train_targets.npy")
    input_valid = np.load(f"{save_dir}/{date}_{dl_number}_valid_features.npy")
    validY = np.load(f"{save_dir}/{date}_{dl_number}_valid_targets.npy")

    input_train = input_train.reshape(input_train.shape[0], input_train.shape[1]*input_train.shape[2])
    input_valid = input_valid.reshape(input_valid.shape[0], input_valid.shape[1]*input_valid.shape[2])


    model_file_path = os.path.join(
        save_dir,
        f"{date}_{dl_number}_{n}th_model.joblib"
    )

    accuracy, loss, auc, coef, model = train_linear_svm_and_save(input_train, trainY)
    val_acc, val_loss, val_auc = eval_on_validation(model, input_valid, validY)
    
    joblib.dump(model, model_file_path)
    np.save(os.path.join(save_dir, f"train_accuracy.npy"), accuracy)
    np.save(os.path.join(save_dir, f"valid_accuracy.npy"), val_acc)
    np.save(os.path.join(save_dir, f"train_loss.npy"), loss)
    np.save(os.path.join(save_dir, f"valid_loss.npy"), val_loss)
    np.save(os.path.join(save_dir, f"train_auc.npy"), auc)
    np.save(os.path.join(save_dir, f"valid_auc.npy"), val_auc)
    np.save(os.path.join(save_dir, f"coef.npy"), coef)

  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 20%|██        | 1/5 [02:21<09:24, 141.06s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 40%|████      | 2/5 [04:41<07:02, 140.74s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of iterations.", ConvergenceWarning)
 60%|██████    | 3/5 [07:03<04:42, 141.17s/it]c:\Users\user\.conda\envs\deepshap\lib\site-packages\sklearn\svm\_base.py:986: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  "the number of